# 9장 웹 UI 추가: 클린 아키텍처의 인터페이스 유연성

파이썬으로 구현하는 클린 아키텍처 - 9장 웹 UI 추가: 클린 아키텍처의 인터페이스 유연성 코드 예제

> **[노트북 참고]** 아래 셀은 노트북 환경에서 `TodoApp` 코드를 import할 수 있도록 경로를 설정합니다. 반드시 첫 번째로 실행해 주세요.

In [ ]:
# ============================================================
# [추가] 노트북 환경 설정 - TodoApp 코드 import를 위한 경로 구성
# Google Colab: 깃허브에서 레포 클론 후 TODOAPP_PATH 자동 설정
# 로컬 환경: 현재 디렉토리의 TodoApp 폴더를 경로로 설정
# 반드시 첫 번째로 실행 필요 (이후 셀들이 이 경로에 의존)
# ============================================================
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/songys/Clean-Architecture-with-Python.git /content/repo
    TODOAPP_PATH = '/content/repo/Chapter_9/TodoApp'
else:
    TODOAPP_PATH = os.path.join(os.getcwd(), 'TodoApp')

if TODOAPP_PATH not in sys.path:
    sys.path.insert(0, TODOAPP_PATH)

## 개요

이 장에서는 작업 관리 시스템을 통해 클린 아키텍처의 핵심 장점 중 하나인 기존 코드 수정 없이 새 인터페이스를 추가할 수 있는 능력을 보여 주려고 한다.

이 장에서 다루는 주요 주제:
* 클린 아키텍처에서의 인터페이스 유연성 이해하기
* 클린 아키텍처에서의 웹 프레젠테이션 패턴
* 플라스크와 클린 아키텍처 통합

### 00_application_container.py

## 애플리케이션 컨테이너: 의존성 역전 원칙의 실제 적용

CLI와 웹 인터페이스가 동일한 컨트롤러에 연결되지만, 핵심 구성 요소는 자신이 어떻게 사용되는지 모른다. 각 필드가 추상 인터페이스로 선언되어 구체적 구현체와 무관하게 동작한다.

In [ ]:
# 애플리케이션 컨테이너: 클린 아키텍처의 모든 구성 요소를 하나로 연결하는 의존성 컨테이너
# 각 필드가 추상 인터페이스 타입으로 선언 → 구체적 구현체를 알지 못함
# 의존성 역전 원칙(DIP)의 핵심 구현체
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리 + TodoApp)
# todo_app/infrastructure/configuration/container.py

# [추가] 노트북에서 실행하기 위한 import
from dataclasses import dataclass
from todo_app.application.repositories.task_repository import TaskRepository
from todo_app.application.repositories.project_repository import ProjectRepository
from todo_app.application.service_ports.notifications import NotificationPort
from todo_app.interfaces.presenters.base import TaskPresenter, ProjectPresenter

@dataclass
class Application:
    """애플리케이션 컨테이너 - 모든 구성 요소를 추상 인터페이스로 연결"""

    task_repository: TaskRepository        # 작업 저장소 인터페이스
    project_repository: ProjectRepository  # 프로젝트 저장소 인터페이스
    notification_service: NotificationPort # 알림 서비스 포트
    task_presenter: TaskPresenter          # 작업 프레젠터 인터페이스
    project_presenter: ProjectPresenter    # 프로젝트 프레젠터 인터페이스

### 01_factory_method.py

## 컴포지션 루트(Composition Root) 패턴

각 구성 요소가 추상 인터페이스(TaskRepository, NotificationPort 등)를 사용하여 선언되는 방식에 주목하자. 이를 통해 각 인터페이스 구현체가 자체적인 의존성을 지닐 수 있으며, 애플리케이션 핵심은 어떤 구현체를 받게 될지 알지 못하게 된다.

In [ ]:
# 컴포지션 루트(Composition Root) 패턴: 팩토리 함수를 통한 의존성 조립
# 인터페이스별 구성 요소(프레젠터)는 외부에서 주입, 핵심 인프라(리포지토리)는 내부 생성
# → 새로운 인터페이스 추가 시 팩토리 호출부만 변경하면 됨
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리 + TodoApp)
# todo_app/infrastructure/configuration/container.py

# [추가] 노트북에서 실행하기 위한 import
from todo_app.infrastructure.configuration.container import Application, create_application
from todo_app.application.service_ports.notifications import NotificationPort
from todo_app.interfaces.presenters.base import TaskPresenter, ProjectPresenter
from todo_app.infrastructure.repository_factory import create_repositories

def create_application(
    notification_service: NotificationPort,
    task_presenter: TaskPresenter,
    project_presenter: ProjectPresenter,
) -> "Application":
    """Application 컨테이너 팩토리 - 인터페이스 독립적 핵심과 인터페이스별 구현체 조합"""
    # 핵심 인프라: 내부에서 생성 (인터페이스와 무관)
    task_repository, project_repository = create_repositories()

    # 컨테이너 조립: 추상 인터페이스에 구체적 구현체 바인딩
    return Application(
        task_repository=task_repository,
        project_repository=project_repository,
        notification_service=notification_service,
        task_presenter=task_presenter,
        project_presenter=project_presenter,
    )

### 02_cli_main.py

## CLI 메인 진입점

CLI 전용 구현체(프레젠터)를 팩토리에 전달하고, 핵심 인프라(리포지토리)는 내부에서 생성된다. 새로운 인터페이스(웹) 추가 시 동일한 패턴으로 웹 전용 프레젠터, 요청 처리, 세션 상태, 오류 표시를 구현하면 된다.

In [ ]:
# CLI 메인 진입점: CLI 인터페이스를 위한 컴포지션 루트
# CLI 전용 구현체(CliTaskPresenter, CliProjectPresenter)를 팩토리에 주입
# 핵심 애플리케이션 코드는 CLI인지 웹인지 알 필요 없음
# Colab에서 click 사용 시: !pip install click (보통 Colab에 기본 설치됨)
# cli_main.py

# [추가] 노트북에서 실행하기 위한 import
from todo_app.infrastructure.configuration.container import create_application
from todo_app.infrastructure.notifications.recorder import NotificationRecorder
from todo_app.interfaces.presenters.cli import CliTaskPresenter, CliProjectPresenter
from todo_app.infrastructure.cli.click_cli_app import ClickCli

def main() -> int:
    """CLI 애플리케이션의 메인 진입점 - CLI 전용 프레젠터 주입"""
    # CLI 전용 구현체를 팩토리에 전달하여 애플리케이션 컨테이너 생성
    app = create_application(
        notification_service=NotificationRecorder(),
        task_presenter=CliTaskPresenter(),       # CLI 전용 프레젠터
        project_presenter=CliProjectPresenter(), # CLI 전용 프레젠터
    )
    # Click CLI 어댑터를 통해 애플리케이션 실행
    cli = ClickCli(app)
    return cli.run()

### 03_handle_create-anti_pattern.py

## 안티패턴: 프레임워크 직접 참조

클린 아키텍처의 효과는 계층 간 명확한 경계를 유지하는 데 달려 있다. 흔한 위반 사례는 개발자가 인터페이스별 형식이 컨트롤러로 스며들도록 할 때 발생해서, 잘못된 방향으로 흐르는 의존성 문제를 발생시킨다.

In [ ]:
# 안티패턴: 컨트롤러에 프레임워크(Click) 직접 참조
# 문제점: 인터페이스 어댑터 계층이 프레임워크 계층(가장 바깥)에 의존
# → 의존성 규칙 위반, CLI 프레임워크 변경 시 컨트롤러 수정 필요
# Colab에서 click 사용 시: !pip install click (보통 Colab에 기본 설치됨)

# [추가] 노트북에서 실행하기 위한 import
import click

def handle_create(self, request_data: dict) -> dict:
    """안티패턴: 컨트롤러에 CLI 전용 형식(click.style) 혼합"""
    try:
        result = self.create_use_case.execute(request_data)
        if result.is_success:
            # 잘못됨: CLI 전용 형식(click.style)이 컨트롤러에 존재
            return {"message": click.style(f"Created task: {result.value.title}", fg="green")}
    except ValueError as e:
        # 잘못됨: CLI 전용 오류 형식이 컨트롤러에 존재
        return {"error": click.style(str(e), fg="red")}

### 04_handle_create_correct.py

## 올바른 생성 처리

프레임워크 독립적인 단순 타입(str)만 수용하고, 모든 형식화를 추상 프레젠터에 위임한다. 이를 통해 CLI/웹/API 등 어떤 인터페이스에서든 동일한 컨트롤러가 동작하며, 프레임워크 교체 시 가장 바깥 계층만 영향받는다.

In [ ]:
# 올바른 패턴: 인터페이스에 구애받지 않는(interface-agnostic) 컨트롤러
# 프레임워크 독립적인 타입(str)만 수용, 형식화는 프레젠터에 위임
# → 동일한 컨트롤러가 CLI/웹/API 등 어떤 인터페이스에서도 동작
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리 + TodoApp)

# [추가] 노트북에서 실행하기 위한 import
from todo_app.interfaces.view_models.base import OperationResult
from todo_app.application.dtos.task_dtos import CreateTaskRequest

def handle_create(self, title: str, description: str) -> OperationResult:
    """올바른 패턴: 프레임워크 독립적 컨트롤러 - 유스케이스와 프레젠터 간 조정 역할"""
    try:
        # 단순 타입(str)으로 요청 수신 → DTO 생성
        request = CreateTaskRequest(title=title, description=description)
        result = self.create_use_case.execute(request)
        if result.is_success:
            # 형식화를 추상 프레젠터에 위임 (CLI/웹 구분 불필요)
            view_model = self.presenter.present_task(result.value)
            return OperationResult.succeed(view_model)

        # 오류도 프레젠터를 통해 형식화
        error_vm = self.presenter.present_error(result.error.message, str(result.error.code.name))
        return OperationResult.fail(error_vm.message, error_vm.code)
    except ValueError as e:
        error_vm = self.presenter.present_error(str(e), "VALIDATION_ERROR")
        return OperationResult.fail(error_vm.message, error_vm.code)

### 05_present_task_cli.py

## CLI 작업 표시

도메인 로직과 웹 표시 요구 사항을 연결하기 위해 웹 관례를 이해하는 프레젠터가 필요하다. 웹 프레젠터의 작동 방식을 이해하려면 먼저 7장의 CLI 프레젠터를 ﻿살펴보자.

In [ ]:
# CLI 프레젠터: 도메인 데이터를 CLI 표시 형식으로 변환
# 대괄호 상태 표시([TODO]), CLI 전용 색상 등 CLI 관례에 맞는 형식 적용
# 7장에서 구현한 CliTaskPresenter의 핵심 메서드
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리 + TodoApp)

# [추가] 노트북에서 실행하기 위한 import
from todo_app.application.dtos.task_dtos import TaskResponse
from todo_app.interfaces.view_models.task_vm import TaskViewModel

def present_task(self, task_response: TaskResponse) -> TaskViewModel:
    """CLI 표시를 위한 작업 데이터 변환 - CLI 관례 적용"""
    return TaskViewModel(
        id=task_response.id,
        title=task_response.title,
        status_display=f"[{task_response.status.value}]",  # CLI 전용: 대괄호 형식
        priority_display=self._format_priority(task_response.priority),  # CLI 전용: 색상 지정
    )

### 06_web_task_presenter.py

## 웹 작업 프레젠터

CLI 프레젠터와 동일한 `TaskPresenter` 인터페이스를 구현하되, 웹 전용 형식(HTML 호환 상태 값, 브라우저용 날짜 형식, 구조화된 완료 정보)을 적용한다. 프레젠터는 도메인 개념이 인터페이스에 어떻게 표시되어야 하는지에 대한 권위 있는 해석자 역할을 한다.

In [ ]:
# 웹 프레젠터: 도메인 데이터를 HTML 표시 형식으로 변환
# CLI 프레젠터와 동일한 TaskPresenter 인터페이스를 구현하되 웹 관례 적용
# HTML 호환 상태 값, 브라우저용 날짜 형식, 구조화된 완료 정보 등
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리 + TodoApp)

# [추가] 노트북에서 실행하기 위한 import
from todo_app.interfaces.presenters.base import TaskPresenter
from todo_app.application.dtos.task_dtos import TaskResponse
from todo_app.interfaces.view_models.task_vm import TaskViewModel

class WebTaskPresenter(TaskPresenter):
    """웹 프레젠터: 동일한 인터페이스, 웹 전용 형식 적용"""

    def present_task(self, task_response: TaskResponse) -> TaskViewModel:
        """웹 표시를 위한 작업 데이터 변환 - HTML 관례 적용"""
        return TaskViewModel(
            id=task_response.id,
            title=task_response.title,
            description=task_response.description,
            status_display=task_response.status.value,  # 웹: HTML 호환 값 (대괄호 없음)
            priority_display=task_response.priority.name,  # 웹: 이름 그대로 사용
            due_date_display=self._format_due_date(task_response.due_date),  # 웹 전용 날짜 형식
            project_display=task_response.project_id,
            completion_info=self._format_completion_info(  # 웹 전용 완료 정보 구조
                task_response.completion_date, task_response.completion_notes
            ),
        )

### 07_format_due_date.py

## 마감일 포맷팅

`_format_due_date`는 시간대 처리, 날짜 형식 문자열, 연체 상태 확인 등 모든 날짜 관련 표시 결정을 캡슐화한다. 도메인 엔터티는 "언제 마감인지"만 관리하고, "어떻게 표시할지"는 프레젠터가 담당한다.

In [ ]:
# 마감일 포맷팅: 프레젠터의 날짜 관련 표시 결정 캡슐화
# 시간대 처리, 날짜 형식 문자열, 연체 상태 확인 등 모든 표시 로직 집중
# 도메인 엔티티는 "언제 마감인지"만 관리, "어떻게 표시할지"는 프레젠터 담당
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리만 사용)
from datetime import datetime, timezone
from typing import Optional


def _format_due_date(self, due_date: Optional[datetime]) -> str:
    """웹 표시용 마감일 포맷 - 연체 여부에 따른 조건부 표시"""
    if not due_date:
        return ""

    # 현재 시간 기준 연체 여부 판단 (비즈니스 규칙이 아닌 표시 결정)
    is_overdue = due_date < datetime.now(timezone.utc)
    date_str = due_date.strftime("%Y-%m-%d")
    # 연체 시 "기한 초과" 접두어 추가
    return f"기한 초과: {date_str}" if is_overdue else date_str

### 08_web_presenter_tests.py

## 웹 프레젠터 테스트

클린 아키텍처의 분리 원칙 덕분에 웹 프레임워크 설정 없이도 연체 날짜 등 복잡한 포맷 시나리오를 테스트할 수 있다.

In [ ]:
# 웹 프레젠터 테스트: 날짜 포맷 변환의 정확성 검증
# 클린 아키텍처의 분리 원칙 덕분에 웹 프레임워크 없이도 포맷 로직 테스트 가능
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리 + TodoApp)
from datetime import datetime, timedelta, timezone

# [추가] 노트북에서 실행하기 위한 import
from todo_app.application.dtos.task_dtos import TaskResponse
from todo_app.domain.value_objects import TaskStatus, Priority
from todo_app.interfaces.presenters.web import WebTaskPresenter


def test_web_presenter_formats_overdue_date():
    """연체 날짜 포맷 검증 - '기한 초과' 접두어 표시 확인"""
    # 준비: 과거 날짜(1일 전)로 TaskResponse 생성
    past_date = datetime.now(timezone.utc) - timedelta(days=1)
    task_response = TaskResponse(
        id="123",
        title="Test Task",
        description="Test Description",
        status=TaskStatus.TODO,
        priority=Priority.MEDIUM,
        project_id="456",
        due_date=past_date,
    )
    presenter = WebTaskPresenter()

    # 실행: 프레젠터를 통한 뷰 모델 생성
    view_model = presenter.present_task(task_response)

    # 검증: "기한 초과" 접두어와 날짜 형식 포함 확인
    assert "기한 초과" in view_model.due_date_display
    assert past_date.strftime("%Y-%m-%d") in view_model.due_date_display


def test_web_presenter_formats_future_date():
    """미래 날짜 포맷 검증 - '기한 초과' 없이 날짜만 표시 확인"""
    # 준비: 미래 날짜(1일 후)로 TaskResponse 생성
    future_date = datetime.now(timezone.utc) + timedelta(days=1)
    task_response = TaskResponse(
        id="123",
        title="Test Task",
        description="Test Description",
        status=TaskStatus.TODO,
        priority=Priority.MEDIUM,
        project_id="456",
        due_date=future_date,
    )
    presenter = WebTaskPresenter()

    # 실행: 프레젠터를 통한 뷰 모델 생성
    view_model = presenter.present_task(task_response)

    # 검증: "기한 초과" 없이 날짜만 표시
    assert "기한 초과" not in view_model.due_date_display
    assert future_date.strftime("%Y-%m-%d") in view_model.due_date_display


# [추가] 테스트 실행
test_web_presenter_formats_overdue_date()
print("test_web_presenter_formats_overdue_date 통과!")

test_web_presenter_formats_future_date()
print("test_web_presenter_formats_future_date 통과!")

### 09_task_entity_antipattern.py

## 안티 패턴: 엔터티에 표시 로직

세션 데이터와 폼 상태는 클린 아키텍처의 경계를 유지하는 데 독특한 작업을 제시한다. 핵심 도메인 로직을 순수하게 유지하면서 시스템이 이런 웹 특유의 관심사를 어떻게 처리하는지 ﻿살펴보자.

In [ ]:
# 안티패턴: 도메인 엔티티가 웹 세션 상태에 직접 접근
# 문제점: Task 엔티티가 웹 컨테이너(세션, 사용자 정보)에 의존
# → 도메인 테스트에 웹 세션 Mock 필요, 도메인 순수성 훼손
# Colab/로컬 모두 동일하게 동작 (Python 표준 라이브러리만 사용)
from datetime import datetime


class Task:
    def complete(self, web_app_contatiner):
        # 잘못됨: Task 도메인 엔티티가 웹 세션에 대해 알면 안 됨
        self.completed_by = web_app_contatiner.user.id  # 웹 전용 사용자 정보 접근
        self.completed_at = datetime.now()

### 10_flask_index_route.py

## Flask 인덱스 라우트

라우트 핸들러는 웹 특유의 상태(세션, 쿼리 파라미터)를 처리하고, HTTP 개념을 도메인 독립적 작업으로 변환하는 경계 역할을 한다. 플래시 메시지, 템플릿 렌더링 등 웹 관심사는 이 외부 계층에만 머문다.

In [ ]:
# Flask 인덱스 라우트: 웹 특유의 상태(세션, 쿼리 파라미터)를 처리하는 외부 계층
# HTTP 개념을 도메인 독립적 작업으로 변환하는 경계 역할
# 플래시 메시지, 템플릿 렌더링 등 웹 관심사는 이 외부 계층에만 존재
# Colab에서 Flask 사용 시: !pip install flask (스텁으로도 코드 구조 학습 가능)
# todo_app/infrastructure/web/routes.py

# [추가] 노트북에서 실행하기 위한 import
try:  # [수정] flask 미설치 시에도 동작하도록 보호 (Colab/로컬 호환)
    from flask import Blueprint, current_app, flash, redirect, render_template, request, url_for
except ImportError:
    class Blueprint:
        def __init__(self, *a, **kw): pass
        def route(self, *a, **kw):
            def decorator(f): return f
            return decorator
    current_app = type('obj', (object,), {'config': {}})()
    def flash(*a, **kw): pass
    def redirect(*a, **kw): pass
    def render_template(*a, **kw): pass
    request = type('obj', (object,), {'args': {}, 'method': 'GET', 'form': {}})()
    def url_for(*a, **kw): return '/'

bp = Blueprint("todo", __name__)

@bp.route("/")
def index():
    """인덱스 라우트 - 웹 전용 상태 처리 후 컨트롤러에 위임"""
    # 컨테이너에서 애플리케이션 객체 획득
    app = current_app.config["APP_CONTAINER"]
    # 웹 전용 쿼리 파라미터 처리 (도메인과 무관)
    show_completed = request.args.get("show_completed", "false").lower() == "true"

    # 컨트롤러에 위임 - HTTP 개념을 도메인 독립적 호출로 변환
    result = app.project_controller.handle_list()
    if not result.is_success:
        error = project_presenter.present_error(result.error.message)
        flash(error.message, "error")  # 웹 전용 사용자 피드백
        return redirect(url_for("todo.index"))

    return render_template("index.html", projects=result.success, show_completed=show_completed)

### 11_flask_new_project_route.py

## Flask 새 프로젝트 라우트

클린 아키텍처의 검증 흐름: 라우트(웹 입력 추출) → 컨트롤러(표준 타입 수신) → 유스케이스(비즈니스 검증). GET 요청은 폼을 렌더링하고, POST 요청은 폼 데이터를 추출하여 컨트롤러에 위임한다.

In [ ]:
# Flask 새 프로젝트 라우트: 폼 제출을 통한 프로젝트 생성
# GET → 폼 렌더링, POST → 폼 데이터 추출 후 컨트롤러에 위임
# 검증 흐름: 라우트(웹 입력 추출) → 컨트롤러(표준 타입 수신) → 유스케이스(비즈니스 검증)
# Colab에서 Flask 사용 시: !pip install flask (스텁으로도 코드 구조 학습 가능)
# todo_app/infrastructure/web/routes.py

# [추가] 노트북에서 실행하기 위한 import
try:  # [수정] flask 미설치 시에도 동작하도록 보호 (Colab/로컬 호환)
    from flask import Blueprint, current_app, flash, redirect, render_template, request, url_for
except ImportError:
    pass  # [보완] 이전 셀에서 이미 스텁 정의됨

bp = Blueprint("todo", __name__)

@bp.route("/projects/new", methods=["GET", "POST"])
def new_project():
    """새 프로젝트 생성 라우트 - HTTP 폼 데이터를 도메인 작업으로 변환"""
    if request.method == "POST":
        # 웹 전용 입력 추출: 폼 데이터에서 이름 획득
        name = request.form["name"]
        app = current_app.config["APP_CONTAINER"]
        # 컨트롤러에 표준 타입(str)으로 위임
        result = app.project_controller.handle_create(name)

        if not result.is_success:
            error = project_presenter.present_error(result.error.message)
            flash(error.message, "error")  # 웹 전용 오류 피드백
            return redirect(url_for("todo.index"))

        project = result.success
        flash(f'프로젝트 "{project.name}" 생성 성공', "success")  # 웹 전용 성공 피드백
        return redirect(url_for("todo.index"))

    # GET 요청: 빈 폼 렌더링
    return render_template("project_form.html")

### 12_web_factory_method.py

## 웹 팩토리 메서드

프레젠테이션 패턴과 상태 관리 방식을 확립했으므로, 이제 플라스크를 클린 아키텍처 시스템에 실질적으로 통합하는 단계로 넘어간다. 앞서 '클린 아키텍처에서의 인터페이스 유연성'에서 ﻿살펴본 애플리케이션 컨테이너 구조를 기반으로, 웹 인터페이스의 플라스크 특화에 집중한다.

In [ ]:
# Flask 앱 팩토리: 플라스크를 클린 아키텍처에 통합하는 설정
# 애플리케이션 컨테이너를 Flask config에 저장하여 라우트에서 접근 가능하게 구성
# 프레임워크 계층(가장 바깥)에서만 Flask에 의존
# Colab에서 Flask 사용 시: !pip install flask (스텁으로도 코드 구조 학습 가능)
# todo_app/infrastructure/web/app.py

# [추가] 노트북에서 실행하기 위한 import
try:  # [수정] flask 미설치 시에도 동작하도록 보호 (Colab/로컬 호환)
    from flask import Flask
except ImportError:
    class Flask:
        """[스텁] flask 미설치 시 사용되는 Flask 스텁 (Colab 호환)"""
        def __init__(self, *a, **kw):
            self.config = {}
        def register_blueprint(self, *a, **kw): pass
        def run(self, *a, **kw): pass
from todo_app.infrastructure.configuration.container import Application

def create_web_app(app_container: Application) -> "Flask":
    """Flask 앱 팩토리 - 클린 아키텍처 컨테이너와 Flask 프레임워크 연결"""
    flask_app = Flask(__name__)
    flask_app.config["SECRET_KEY"] = "dev"  # 프로덕션에서는 환경 변수로 대체 필요
    flask_app.config["APP_CONTAINER"] = app_container  # 컨테이너를 Flask config에 저장

    # 블루프린트 등록 - 라우트 핸들러 연결
    # from . import routes  # [수정] 노트북에서는 상대 import 불가
    # flask_app.register_blueprint(routes.bp)

    return flask_app

### 13_click_create_task.py

## Click 작업 생성 명령

CLI의 `click.prompt()`와 웹의 `request.form`은 동일한 아키텍처 패턴의 인터페이스별 구현이다. 입력 수집 후 컨트롤러에 표준 타입(str)으로 위임한다.

In [ ]:
# Click CLI 작업 생성 명령: CLI 인터페이스의 입력 수집 방식
# click.prompt()를 통해 CLI 전용 입력 수집 후 컨트롤러에 표준 타입으로 위임
# 웹 라우트의 request.form과 동일한 역할 - 인터페이스별 입력 추출
# Colab에서 click 사용 시: !pip install click (보통 Colab에 기본 설치됨)
# todo_app/infrastructure/cli/click_cli_app.py

# [추가] 노트북에서 실행하기 위한 import
import click

def _create_task(self):
    """CLI 작업 생성 - CLI 전용 입력 수집 후 컨트롤러에 위임"""
    # CLI 전용 입력 방식: 대화형 프롬프트 (웹의 폼 제출에 대응)
    title = click.prompt("작업 제목", type=str)
    description = click.prompt("설명", type=str)
    # 컨트롤러에 표준 타입(str)으로 위임 - CLI/웹 구분 없는 동일한 호출
    result = self.app.task_controller.handle_create(title=title, description=description)

### 14_flask_new_task_route.py

## Flask 새 작업 라우트

웹 라우트는 CLI와 동일한 아키텍처 패턴이되, HTTP 요청-응답 주기에 맞게 조정된다. `project_id`는 URL에서, 작업 세부 정보는 폼 필드에서 추출하여 컨트롤러에 위임한다.

In [ ]:
# Flask 새 작업 라우트: URL 매개변수와 폼 데이터를 조합한 작업 생성
# project_id는 URL(/projects/<project_id>/tasks/new)에서, 나머지는 폼에서 추출
# CLI의 click.prompt()와 동일한 아키텍처 패턴 - 인터페이스만 다름
# Colab에서 Flask 사용 시: !pip install flask (스텁으로도 코드 구조 학습 가능)

# [추가] 노트북에서 실행하기 위한 import
try:  # [수정] flask 미설치 시에도 동작하도록 보호 (Colab/로컬 호환)
    from flask import Blueprint, current_app, flash, redirect, render_template, request, url_for
except ImportError:
    pass  # [보완] 이전 셀에서 이미 스텁 정의됨

bp = Blueprint("todo", __name__)

@bp.route("/projects/<project_id>/tasks/new", methods=["GET", "POST"])
def new_task(project_id):
    """새 작업 생성 라우트 - URL 파라미터 + 폼 데이터를 컨트롤러에 위임"""
    if request.method == "POST":
        app = current_app.config["APP_CONTAINER"]
        # 웹 전용 입력 추출: URL 매개변수(project_id) + 폼 필드(title, description 등)
        result = app.task_controller.handle_create(
            project_id=project_id,                    # URL에서 추출
            title=request.form["title"],              # 폼 데이터에서 추출
            description=request.form["description"],  # 폼 데이터에서 추출
            priority=request.form["priority"],        # 폼 데이터에서 추출
            due_date=request.form["due_date"] if request.form["due_date"] else None,  # 선택적 필드
        )

        if not result.is_success:
            error = task_presenter.present_error(result.error.message)
            flash(error.message, "error")
            return redirect(url_for("todo.index"))

        task = result.success
        flash(f'작업 "{task.title}" 생성 성공', "success")
        return redirect(url_for("todo.index"))

    # GET 요청: 작업 생성 폼 렌더링
    return render_template("task_form.html", project_id=project_id)

### 16_web_main.py

## 웹 메인 진입점

CLI 진입점과 동일한 컴포지션 루트 구조이되, 웹 전용 프레젠터(`WebTaskPresenter`, `WebProjectPresenter`)를 팩토리에 주입한다. 구체적 구현체 선택은 런타임에 결정된다.

In [ ]:
# 웹 메인 진입점: 웹 인터페이스를 위한 컴포지션 루트
# CLI 진입점과 동일한 구조이되 웹 전용 프레젠터(WebTaskPresenter, WebProjectPresenter) 주입
# 의존성 역전 원칙: 구체적 구현체 선택은 런타임 시점에 결정
# Colab에서 Flask 사용 시: !pip install flask (스텁으로도 코드 구조 학습 가능)

# [추가] 노트북에서 실행하기 위한 import
from todo_app.infrastructure.configuration.container import create_application
from todo_app.infrastructure.notifications.recorder import NotificationRecorder
from todo_app.interfaces.presenters.web import WebTaskPresenter, WebProjectPresenter
try:  # [수정] flask 미설치 시에도 동작하도록 보호 (Colab/로컬 호환)
    from todo_app.infrastructure.web.app import create_web_app
except (ImportError, ModuleNotFoundError):
    def create_web_app(app_container):
        """[스텁] flask 미설치 시 사용되는 create_web_app 스텁 (Colab 호환)"""
        return None

def main():
    """웹 애플리케이션 진입점 - 웹 전용 프레젠터를 팩토리에 주입"""
    # CLI와 동일한 팩토리 호출, 웹 전용 구현체만 다름
    app_container = create_application(
        notification_service=NotificationRecorder(),
        task_presenter=WebTaskPresenter(),       # 웹 전용 프레젠터
        project_presenter=WebProjectPresenter(), # 웹 전용 프레젠터
    )

    # Flask 앱 생성 및 실행
    flask_app = create_web_app(app_container)
    flask_app.run(debug=True)


if __name__ == "__main__":
    main()